# Omnipose — extracción de confianza por máscara (config elegida)

Corre **solo** la configuración global de la tesis (`cyto2_omni`, `mask_threshold=-2`,
`flow_threshold=0.4`) y extrae, por cada instancia predicha, la **confianza** del
segmentador (media del mapa `cellprob`/distancia de Omnipose = `flows[2]`) — análoga
al `prob_mean` de MoDL — para el filtro de spurious.

> Flujo en Colab: 1) correr la celda de instalación, 2) **Runtime → Restart session**,
> 3) montar Drive, 4) correr la extracción.

In [ ]:
# Instalación (igual que el notebook original). Reiniciar sesión después.
!pip uninstall -y omnipose cellpose_omni ncolor
!pip cache remove cellpose_omni
!pip install --no-cache-dir omnipose "ncolor<2"
print("\n>>> Runtime -> Restart session, y seguí con drive.mount <<<")

**Reiniciá la sesión antes de continuar** (Runtime → Restart session).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import numpy as np
import pandas as pd
from cellpose_omni import io, models

# --- RUTAS -------------------------------------------------------------------
# IMPORTANTE: apuntá IMAGES_DIR a las imágenes cuyas máscaras usás en el estudio.
#   - 9 frames originales (lo analizado hasta ahora): carpeta Images del run ORIGINAL.
#   - 10 frames extendidos: Images_ext.
IMAGES_DIR = "/content/drive/MyDrive/Tesis Santiago Feliu/Notebooks/Cellpose/Images"      # <-- AJUSTAR
OUTPUT_DIR = "/content/drive/MyDrive/Tesis Santiago Feliu/Notebooks/Omnipose/confidence"  # <-- salida
os.makedirs(OUTPUT_DIR, exist_ok=True)
MAPS_DIR = os.path.join(OUTPUT_DIR, "cellprob_maps")
os.makedirs(MAPS_DIR, exist_ok=True)

# --- CONFIGURACIÓN ELEGIDA (única) ------------------------------------------
MODEL_NAME = "cyto2_omni"
MASK_THRESHOLD = -2
FLOW_THRESHOLD = 0.4
MIN_AREA = 16  # coherente con el resto del estudio

EVAL_PARAMS = dict(
    omni=True, cluster=True, niter=None, channels=None,
    resample=True, affinity_seg=False, verbose=False, tile=False,
)


def get_tif_files(directory):
    return {f: os.path.join(directory, f)
            for f in sorted(os.listdir(directory)) if f.endswith(".tif")}


def cargar_imagen(path):
    img = io.imread(path)
    if img.ndim == 3:
        img = img.mean(axis=-1)
    return img

In [ ]:
# --- CORRIDA -----------------------------------------------------------------
model = models.CellposeModel(gpu=True, model_type=MODEL_NAME)
rows = []
images = get_tif_files(IMAGES_DIR)
print(f"{len(images)} imagenes | config {MODEL_NAME} mt={MASK_THRESHOLD} ft={FLOW_THRESHOLD}")

for image_name, path in images.items():
    stem = image_name[:-4]
    img = cargar_imagen(path)

    # Capturamos flows (antes se descartaba). flows[2] = cellprob / campo de distancia.
    masks, flows, styles = model.eval(
        img, mask_threshold=MASK_THRESHOLD, flow_threshold=FLOW_THRESHOLD, **EVAL_PARAMS
    )
    cellprob = np.asarray(flows[2], dtype=float)
    assert cellprob.shape == masks.shape, \
        f"cellprob {cellprob.shape} != masks {masks.shape} (revisar indice de flows)"

    np.save(os.path.join(MAPS_DIR, f"{stem}_pred.npy"), masks)
    np.save(os.path.join(MAPS_DIR, f"{stem}_cellprob.npy"), cellprob)

    for lab in np.unique(masks):
        if lab == 0:
            continue
        m = masks == lab
        area = int(m.sum())
        if area < MIN_AREA:
            continue
        cp = cellprob[m]
        rows.append({
            "frame": stem, "label": int(lab), "area_px": area,
            "cellprob_mean": float(cp.mean()),
            "cellprob_median": float(np.median(cp)),
            "cellprob_p10": float(np.percentile(cp, 10)),
            "cellprob_min": float(cp.min()),
            "cellprob_std": float(cp.std()),
        })
    print(f"  {stem}: {int(masks.max())} instancias")

df = pd.DataFrame(rows)
out_csv = os.path.join(OUTPUT_DIR, "omnipose_confidence_cyto2_mt-2_ft0.4.csv")
df.to_csv(out_csv, index=False)
print(f"\nGuardado: {out_csv}  ({len(df)} instancias)")
print("Maps en:", MAPS_DIR)
df.head()